#### Vision 모델 내부 동작 원리
- 이미지 인코딩
- 프로젝션: 이미지 벡터를 LLM 토큰 공간으로 변호나
- 토큰 결합: 시각 토큰 + 텍스트 토큰을 순서대로 배열
- LLM 처리
- 텍스트 생성

In [9]:
import os
import time
import base64
from dotenv import load_dotenv
from pathlib import Path

from langchain_ibm import WatsonxEmbeddings
from langchain_ibm import ChatWatsonx
from langchain_ollama import ChatOllama

from langchain_core.prompts import  ChatPromptTemplate
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_chroma import Chroma

In [2]:
load_dotenv()

apiKey = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_ai_url = os.getenv("WATSONX_URL")
hf_token = os.environ["HF_TOKEN"]
COHERE_API_KEY = os.environ["COHERE_API_KEY"]
SERPER_API_KEY = os.getenv("SERPER_API_KEY")

watson_embedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url = f"{watsonx_ai_url}",
    api_key = f"{apiKey}",
    project_id=f"{project_id}"
)

watson_llm = ChatWatsonx(
  model_id="ibm/granite-4-h-small",
  url=f"{watsonx_ai_url}",
  api_key = f"{apiKey}",
  project_id=f"{project_id}",
  max_tokens = 2000,
  params = {
    "temperature":0
  }
)

qwen_llm = ChatOllama(model="qwen3.5:4b", temperature=0)

# vision 모델
vision_llm = ChatOllama(model="minimax-m3:cloud", temperature=0)

parser = StrOutputParser()

#### 1. 로컬 이미지 처리

In [6]:
# image를 base64 인코딩 - rb로 읽어야함

def encode_image(image_path:str)->str:
  with open(image_path, "rb") as f:
    return base64.b64encode(f.read()).decode("utf-8")

In [6]:
# 이미지 + 텍스트: 이미지 기반으로 질문
def ask_about_image(image_path, question):
  image_base64 = encode_image(image_path)
  prompt = ChatPromptTemplate.from_messages(
    [
      (
        "human", [
          {"type":"image_url", "image_url":{'url': f'data:image.jpeg;base64,{image_base64}'}},
          {"type":"text", "text":question}
        ]
      )
    ]
  )
  
  chain = prompt | vision_llm | parser
  response = chain.invoke({"image_base64":image_base64, "question": question})
  return response


In [8]:
ask_about_image("./image/animals1.jpg", "이게 뭐야?")

"이건 고양이 사진이에요! 🐱\n\n구체적으로 보면:\n- **고양이 종류**: 갈색 줄무늬가 있는 짧은 털을 가진 고양이입니다. '브라운 태비(Brown Tabby)' 패턴으로 보입니다.\n- **특징**: \n  - 눈이 아주 선명한 **민트색/연한 청록색**이에요\n  - 카메라를 올려다보는 구도로 찍은 사진\n  - 입이 살짝 벌어져 있고 긴 수염이 인상적입니다\n- **배경**: 따뜻한 톤의 나무 바닥 위에서 촬영된 것 같아요\n\n아주 귀엽고 표정이 인상적인 고양이네요! 혹시 다른 궁금한 점이 있으신가요? 😊"

#### 2.URL 이미지

In [14]:
# 이미지 다운로드
import requests
from io import BytesIO # 저장
from PIL import Image


url = "https://cdn.aitimes.kr/news/photo/202511/37466_56875_3234.jpg"
img = requests.get(url).content

# 인코딩
img_b64 = base64.b64encode(img).decode()

# 모델
message = HumanMessage(
  content=[
    {"type":"image_url", "image_url":{'url': f'data:image.jpeg;base64,{img_b64}'}},
    {"type":"text", "text":"로봇이 뭐하고있니?"}
  ]
)

result = vision_llm.invoke([message]).content
print(result)

이미지를 보면, 로봇은 한 여성을 도와주고 있는 모습입니다.

**로봇의 활동:**
- 키가 크고 은색/회색의 휴머노이드 로봇이 여성 옆에 서 있습니다
- 로봇의 가슴 부분에 파란색 LED 표시등이 켜져 있어 활성화된 상태임을 보여줍니다
- 로봇이 **태블릿을 들고 여성에게 보여주거나** 정보를 공유하고 있는 것으로 보입니다
- 함께 업무를 협력하는 모습입니다

**장면 전체:**
- 여성은 흰색 블라우스를 입고 태블릿을 들고 있으며, 미소 짓고 있습니다
- 앞쪽에는 노트북이 놓여 있고, 서류 클립보드, 안경 등 사무용품이 있습니다
- 밝고 현대적인 사무실 환경처럼 보입니다

전체적으로 **로봇과 인간이 협력하여 업무를 수행하는 장면**을 보여주는 이미지로, AI 어시스턴트나 협업 로봇(co-bot)이 작업자를 지원하는 컨셉의 사진으로 보입니다. 😊


#### 3. OCR

In [24]:
# 모든 이미지를 jpeg 저장 / 인코딩
def pil_to_base64(img:Image.Image,format="JPEG")->str:
  buffer = BytesIO()
  img.save(buffer, format=format)
  return base64.b64encode(buffer.getvalue()).decode('utf-8')

In [25]:
# 로컬에 있는 이미지 읽어오기
img = Image.open("./image/system.png")

# 리사이징
image_resized = img.resize((800,600))

# gray
img_gray = image_resized.convert("L")
img_b64 = pil_to_base64(img_gray)

message = HumanMessage(
  content=[
    {"type":"image_url", "image_url":{'url': f'data:image.jpeg;base64,{img_b64}'}},
    {"type":"text", "text":"이 문서의 텍스트를 추출해주세요"}
  ]
)

result = vision_llm.invoke([message]).content
print(result)

# 문서 텍스트 추출

## 미래로봇추진단(서울 근무)
## S/W개발 - 시스템 소프트웨어

---

### 포지션 소개 (Job Overview)
휴머노이드 로봇의 실시간 제어 시스템 및 소프트웨어 플랫폼을 개발하는 직무입니다. 운용체제 환경 구성, 디바이스 드라이버, 실시간 제어 프레임워크 등 로봇 동작의 핵심 기반이 되는 소프트웨어를 설계 및 개발하며, 하드웨어 설계 조직 및 AI 연구 조직과 긴밀히 협업합니다.

### 수행업무 (Job Details)
- 휴머노이드 로봇의 실시간 제어 프레임워크를 설계하고 개발합니다.
- 로봇, 센서 등 로봇 하드웨어 제어를 위한 디바이스 드라이버 및 하드웨어 추상화 계층을 개발합니다.
- 로봇 제어를 운영체제(Linux 기반 실시간 OS 등) 환경을 구성하고 시스템 성능을 최적화합니다.
- 유관 부서와 협업하여 로봇 조직 등 시 기능과 제어 시스템 간 연동 미들웨어를 개발합니다.

### 자격요건 (Requirements)
- 컴퓨터, 전기·전자, 기계, 로봇공학 등 관련 전공을 하신 분
- 운영체제 기반 개념에 대한 이해도를 보유하신 분
- 요구사항을 분석하여 소프트웨어를 구조적으로 설계 및 구현하는 역량을 보유하신 분
- 다양한 분야의 엔지니어와 적극적으로 소통하며 협업할 수 있는 역량을 보유하신 분

### 우대사항 (Preferences)
- C/C++ 기반 시스템 프로그래밍 역량을 보유하신 분
- Linux 기반 임베디드 시스템 개발 경험을 보유하신 분 (Kernel, Device Driver, BSP 등)
- CAN, EtherCAT, UDP 등 하드웨어 통신 프로토콜 활용 경험을 보유하신 분
- ROS2 기반 로봇제어 소프트웨어 수행 경험을 보유하신 분
- Git 기반 협업 및 CI/CD 환경에서의 개발 경험을 보유하신 분

### 커리어 비전 (Career Vision)
휴머노이드 로봇의 초기 개발 단계부터 참여하여 핵심 개발자로 성장할 수 있습니다. 실시간 제어, 시스템 아키텍처, 로봇 미들웨어 등

In [28]:
# 다중 이미지 비교분석

def compare_images(image_paths, question):
  content=[]
  
  for i, path in enumerate(image_paths, 1):
    img_b64 = encode_image(path)
    content.append({"type":"image_url", "image_url":{'url': f'data:image.jpeg;base64,{img_b64}'}},)
    content.append({"type":"text", "text":f'[이미지 {i}]'})
  
  # 질문 추가
  content.append({"type":"text", "text":question})
  message = HumanMessage(content=content)
  response = vision_llm.invoke([message])
  return response.content

In [30]:
result = compare_images(['./image/chart1.png','./image/chart2.png'], "두 차트를 비교하여 주요 변화 수치를 해주세요")
print(result)

# 📊 두 차트 비교 분석

## 📈 차트 1: 최근 6개월 전체 화장품 수출 추이

| 기간 | 수출액(백만$) | YoY 증감률 |
|------|-------------|-----------|
| 25년 12월 | 883.7 | +10.2% |
| 26년 1월 | 841.6 | **+33.7%** (최고) |
| 26년 2월 | 752.1 | +1.7% (최저) |
| 26년 3월 | 960.4 | +23.0% |
| 26년 4월 | **1,096.3** (최고 수출액) | +21.7% |
| 26년 5월(1~20일) | 671.1 | **-16.0%** |

## 🌍 차트 2: 5월(1~20일) 주요국 일평균 수출액

| 국가 | 일평균 수출액(백만$) | YoY 증감률 |
|------|---------------------|-----------|
| 중화권(중국) | **13.13** (1위) | **+76.4%** (최고) |
| 미국 | 11.66 | +40.3% |
| 유럽 | 11.14 | +61.3% |
| 동남아 | 4.91 | +22.0% |
| 일본 | 약 3.1 | **-14.1%** (유일 마이너스) |

---

## 🔍 주요 변화 수치 요약

### 1️⃣ **5월 수출 급락의 원인 = 일본 시장을 제외하면 성장세 지속**
- 전체 5월(1~20일) YoY **-16%**
- 그러나 일본을 제외한 4개 지역은 **모두 +22% 이상 성장**
- 5월 급락은 **일본(-14.1%)** 단일 시장에 기인

### 2️⃣ **중국 시장이 전체 흐름을 견인**
- 중화권 YoY **+76.4%** → 최고 성장률
- 수출액 규모도 **1위(13.13백만$)** → 실질적 성패를 좌우

### 3️⃣ **수출액 절대 수치는 5월까지 상승 후 급락**
- 25년 12월 883.7 → 26년 4월 **1,096.3** (▲약 +24%)
- 그러나 5월(1~20일) **671.1로 급감** (4월 대비 약 -39% 수준, 기간차이 감안 필요)

In [29]:
result = compare_images(['./image/fridge.jpg','./image/table.jpg'], "두 제품의 차이점을 비교 분석해주세요")
print(result)

# 두 제품 비교 분석

## 🖼️ 이미지 1: 4도어 프렌치도어 냉장고

### 제품 특징
- **형태**: 4도어 프렌치도어 스타일 (상단 2도어 + 하단 2도어)
- **컬러**: 상단 크림/아이보리, 하단 화이트 (투톤 디자인)
- **구분선**: 중앙에 블랙 가로 손잡이 바
- **디자인**: 깔끔하고 모던한 미니멀 스타일

### 장점
- ✅ 넉넉한 용량 (패밀리용)
- ✅ 상하 분리 보관으로 에너지 효율 우수
- ✅ 도어 부분 개방으로 냉기 손실 최소화
- ✅ 세련된 투톤 컬러로 주방 인테리어 연출

---

## 🖼️ 이미지 2: 마블 패턴 식탁/테이블

### 제품 특징
- **소재**: 천연/인조 대리석 상판 (화이트 베이스 + 회색 veins)
- **프레임**: 블랙 + 골드 포인트 다리
- **형태**: 모서리 라운드 처리된 직사각형
- **사이즈**: 4인~6인용으로 추정

### 장점
- ✅ 고급스러운 마블 무늬로 럭셔리한 분위기
- ✅ 모서리 라운딩으로 안전성 확보
- ✅ 골드 다리로 모던하고 세련된 디자인
- ✅ 내구성 우수, 관리가 비교적 쉬움

---

## 📊 핵심 차이점 비교

| 항목 | 냉장고 (이미지 1) | 식탁 (이미지 2) |
|------|------------------|----------------|
| **용도** | 식료품 보관 (가전) | 식사/작업 공간 (가구) |
| **소재** | 금속/플라스틱 | 대리석+금속 프레임 |
| **컬러** | 크림+화이트 | 화이트+블랙+골드 |
| **스타일** | 모던 미니멀 | 모던 럭셔리 |
| **설치 위치** | 주방 | 거실/식당 |
| **동작 방식** | 전기 작동 | 수동 |
| **유지보수** | 정기 점검 필요 | 표면 청소 정도 |

---

## 🎯 결론

두 제품은 **완전히 다른 카테고리**의 제품입니다.

- **냉장고**는 🏠 **주방 가전**으로, 기능성과 보관이 주목적
- **식탁**은 🪑 **생활 가구**로, 공간 연출과 사용성

#### 멀티모달
- 이미지 기반 문서 분석 시스템

In [7]:
# 이미지 -> 설명문 생성

def image_to_caption(image_path):
  img_b64 = encode_image(image_path)
  message = HumanMessage(
    content = [
      {"type":"image_url", "image_url":{'url': f'data:image.jpeg;base64,{img_b64}'}},
      {"type":"text", "text":"""
       이 이미지를 검색용 설명문으로 요약하세요.
       200자 이내로 작성하세요
       핵심 객체와 텍스트만 포함하세요
       """}
    ]
  )
  
  return vision_llm.invoke([message]).content

In [8]:
caption = image_to_caption("./image/chart1.png")
print(caption)

최근 6개월 화장품 수출액 추이 차트. 파란색 막대는 수출액(백만$), 빨간색 꺾은선은 전년동기대비(YoY) 성장률을 나타냄. 25년 12월 883.7(10.2%), 26년 1월 841.6(33.7%), 2월 752.1(1.7%), 3월 960.4(23%), 4월 1096.3(21.7%)으로 증가하다 5월 1~20일 671.1(-16%)으로 급감.


In [ ]:
# Document

def build_multimodal_index(image_dir:str, text_docs:list[Document]):
  all_docs = list(text_docs)
  
  # image_dir 안 파일 가져오기
  image_files = list(Path(image_dir).glob("*.{jpg, jpeg, png, gif}"))
  
  # caption 생성 => Document => 임베딩
  for img_path in image_files:
    caption = image_to_caption(img_path=img_path)
    
    doc = Document(page_content=caption, metadata={
      "source": str(img_path),
      "type":"image",
      "image_path":str(img_path)
    })
    all_docs.append(doc)

  return Chroma.from_documents(all_docs, watson_embedding, persist_directory="./db/multimodal_db")

In [ ]:
# 검색결과에 이미지 포함 여부 확인
def search_with_images(vectorstore, query):
  results = vectorstore.similarity_search(query, k=5)
  text_results = [r for r in results if r.metadata.get("type") != "image"]
  image_results = [r for r in results if r.metadata.get("type") == "image"]
  
  print(f"텍스트 결과 {len(text_results)}개, 이미지 결과{len(image_results)}개")
  
  return text_results, image_results

In [ ]:
image_doc = Document(
  page_content=caption,
  metadata={"type":"image","image_path":"./image/chart1.png"}
)

docs = [
  Document(
    page_content="2025년 매출은 증가했다"
  ),
  image_doc
]

build_multimodal_index("./image", docs)

In [13]:
vectorstore = Chroma(embedding_function=watson_embedding, persist_directory="./db/multimodal_db")

results = vectorstore.similarity_search("매출 추세", k=3)

for r in results:
  print(r.page_content)

2020년 매출은 증가했다
최근 6개월 화장품 수출액 추이 차트. 파란색 막대는 수출액(백만$), 빨간색 꺾은선은 전년동기대비(YoY) 성장률을 나타냄. 25년 12월 883.7(10.2%), 26년 1월 841.6(33.7%), 2월 752.1(1.7%), 3월 960.4(23%), 4월 1096.3(21.7%)으로 증가하다 5월 1~20일 671.1(-16%)으로 급감.


In [ ]:
def multimodal_answer(vectorstore, question):
  text_results, image_results = search_with_images(vectorstore=vectorstore, query=question)
  
  # 텍스트 결과 하나의 컨텍스트로 생성
  text_context = "\n\n".join(r.page_content for r in text_results)
  
  referenced_images = []
  # 이미지로 검색된 경우
  image_context = ""
  for img_doc in image_results[:3]:
    img_path = img_doc.metadata.get("image_path")
    
    img_b64 = encode_image(img_path)
    analysis = vision_llm.invoke([
      HumanMessage(
        content=[
          {"type":"image_url", "image_url":{'url': f'data:image.jpeg;base64,{img_b64}'}},
          {"type":"text", "text":"""이 이미지에서 다음 질문과 관련된 내용을 설명하세요: {question}"""}
        ]
      )
    ]).content
    
    image_context += f"[이미지 분석: {img_path}]\n{analysis}\n\n"
    referenced_images.append(img_path)

  # 최종 답변
  combined_context = text_context + "\n\n" + image_context

  final_prompt = ChatPromptTemplate.from_messages([
    ('system', '다음 텍스트와 이미지 분석 결과를 참고하여 질문에 답하세요\n\n{context}'),
    ('human', '{question}')
  ])

  parser = StrOutputParser()
  chain = final_prompt | watson_llm | parser

  answer = chain.invoke({
    "context" : combined_context,
    "question": question
  })

  return {"answer": answer, "images":referenced_images}

In [19]:
multimodal_answer(vectorstore, "최근 6개월 전체 화장품 수출액?")

텍스트 결과 1개, 이미지 결과1개


{'answer': '최근 6개월 동안의 한국 화장품 수출액은 다음과 같습니다:\n\n- 2025년 12월: 883.7 백만 달러\n- 2026년 1월: 841.6 백만 달러\n- 2026년 2월: 752.1 백만 달러\n- 2026년 3월: 960.4 백만 달러\n- 2026년 4월: 1,096.3 백만 달러\n- 2026년 5월(1~20일): 671.1 백만 달러\n\n이 데이터는 최근 6개월 동안의 수출 추이를 보여줍니다.',
 'images': ['./image/chart1.png']}